![](/Types__of_streaming.png)

![](/Volumes/workspace/monschema1/images/streaming_trigger_types.JPG)

![](/Volumes/workspace/monschema1/images/Streaming_output_mode.png)

In [0]:
from pyspark.sql.functions import col, approx_count_distinct, count
batch_df = (spark.read.format("csv") 
            .option("delimiter", ";")
            .option("header","true")
            .load('/Volumes/workspace/monschema1/streaming/generated_data.csv')
            .filter(col("traffic_source") == "email")
            .withColumn("mobile",col("device").isin(["ios","android"]))
      
            
            .select("user_id",col("event_timestamp").cast("BIGINT").alias("event_timestamp"),"mobile"))

print(batch_df.isStreaming)

display(batch_df)


In [0]:
from pyspark.sql.functions import col, approx_count_distinct, count

df=(spark.read.format("csv") 
            .option("delimiter", ";")
            .option("header","true")  
            .load('/Volumes/workspace/monschema1/streaming/generated_data.csv'))



streaming_df = (spark.readStream.format("csv") 
            .option("delimiter", ";")
            .option("header","true")
            .schema(df.schema)   
            .load('/Volumes/workspace/monschema1/streaming')
            .filter(col("traffic_source") == "email")
            .withColumn("mobile",col("device").isin(["ios","android"]))
      
            
            .select("user_id",col("event_timestamp").cast("BIGINT").alias("event_timestamp"),"mobile"))

print(streaming_df.isStreaming)

display(streaming_df, streamName="display_user_devices", checkpointLocation="/Volumes/workspace/monschema1/streaming/checkpoints/display_user_devices")


In [0]:
checkpoint_path = "/Volumes/workspace/monschema1/streaming/checkpoints/email_traffic"
device_query = (streaming_df.writeStream
                .outputMode("append")
                .format("delta")
                .queryName("email_traffic")
                .trigger(availableNow=True)
                .option("checkpointLocation", checkpoint_path)
                .toTable("email_traffic_only")
                )

In [0]:
device_query.name

In [0]:
device_query.status

In [0]:
device_query.lastProgress

In [0]:
import time
time.sleep(10)
device_query.stop()


In [0]:
device_query.status

In [0]:
device_query.awaitTermination()

In [0]:
from pyspark.sql.functions import col, approx_count_distinct, count

df=(spark.readStream.format("csv") 
            .option("delimiter", ";")
            .option("header","true")  
            .load('/Volumes/workspace/monschema1/my_volume/Stream_window/generated_100_rows.csv'))


In [0]:
from pyspark.sql.types import StructType
from pyspark.sql.functions import from_json, col,sum, window

ecommerce_schema = StructType.fromDDL("""
  purchase_revenue_in_usd DOUBLE,
  total_item_quantity BIGINT,
  unique_items BIGINT
""")
 

In [0]:
from pyspark.sql.functions import col, parse_json

stream_df = (
    df.withColumn("ecommerce", from_json(col("ecommerce"), ecommerce_schema))
      .withColumn("geo", from_json(col("geo"), StructType.fromDDL("city STRING, state STRING")))
      .withColumn("event_timestamp", col("event_timestamp").cast("timestamp"))
      .withColumn("event_previous_timestamp", col("event_previous_timestamp").cast("timestamp"))
      .filter("ecommerce.purchase_revenue_in_usd IS NOT NULL AND ecommerce.purchase_revenue_in_usd != 0")
)
 

In [0]:
print(stream_df.isStreaming)


In [0]:
windowed_df = (
    stream_df.withWatermark(eventTime = "event_timestamp", delayThreshold = "7 DAYS")
        .groupBy(
            window(timeColumn="event_timestamp", windowDuration="5 DAYS" ),
            "device","geo.city")
           .agg(sum("ecommerce.purchase_revenue_in_usd").alias("total_revenue"))
)

In [0]:
display (windowed_df.select("window","city","total_revenue")
         .filter("window.start >= '2020-01-01' AND window.start <= '2022-01-31'"))




In [0]:
print(windowed_df.isStreaming)

In [0]:
checkpoint_path="/Volumes/workspace/monschema1/streaming/checkpoints/query_revenue_by_city_by_hour_append"

windowed_query= (windowed_df.writeStream 
                 .queryName("query_revenue_by_city_by_hour_append")
                 .option("checkpointLocation",checkpoint_path)
                 .trigger(availableNow=True)
                 .outputMode("append")
                 .table("monschema1.revenue_by_city_by_hour_append")                 
                 )



In [0]:
%fs ls dbfs:/user/hive/warehouse/

In [0]:
%sql
DESCRIBE DATABASE monschema1